In [ ]:
# import kagglehub
# 
# # Download latest version
# path = kagglehub.dataset_download("seryouxblaster764/fgvc-aircraft")
# 
# print("Path to dataset files:", path)

In [1]:
from torch.utils.data import Dataset
from PIL import Image
import os
import pandas as pd
from torchvision.models import resnet18
from AircraftDataset import AircraftDataset

In [2]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class AircraftCNN(nn.Module):
    def __init__(self, num_classes):
        super(AircraftCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        self.pool = nn.MaxPool2d(2, 2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))  # гарантированный выход 4×4

        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # -> [B, 32, H/2, W/2]
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # -> [B, 64, H/4, W/4]
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # -> [B, 128, H/8, W/8]
        x = self.adaptive_pool(x)  # -> [B, 128, 4, 4]
        x = x.view(x.size(0), -1)  # -> [B, 2048]
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x


In [4]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total


In [5]:
import torch
from torch import nn, optim
from tqdm import tqdm
import wandb


def train_model(model, train_loader, val_loader, device, epochs=10):
    # wandb.init(project="aircraft-classifier",
    #            config={"epochs": epochs,
    #                    "lr": 1e-4,
    #                    "batch_size": train_loader.batch_size}
    #            )
    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_acc = 0.0
    patience = 3
    counter = 0
    
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}"):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total
        val_acc = evaluate_model(model, val_loader, device)

        # wandb.log({
        #     "train_loss": running_loss / total,
        #     "train_acc": train_acc,
        #     "val_acc": val_acc,
        #     "epoch": epoch + 1
        # })

        if val_acc > best_acc:
            best_acc = val_acc
            counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print("✅ Model saved.")
            # wandb.run.summary["best_val_acc"] = best_acc
        else:
            counter += 1
            if counter >= patience:
                print('Early stopping triggered')
                break        
        scheduler.step()
        
        print(f"Epoch {epoch + 1}: Loss={running_loss / total:.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")


### загрузим данные

In [6]:
train_small_dataset = AircraftDataset('data/fgvc-aircraft/train_small.csv', 'data/fgvc-aircraft/images', transform)
val_small_dataset = AircraftDataset('data/fgvc-aircraft/val_small.csv', 'data/fgvc-aircraft/images', transform)
test_small_dataset = AircraftDataset('data/fgvc-aircraft/test_small.csv', 'data/fgvc-aircraft/images', transform)

train_small_loader = DataLoader(train_small_dataset, batch_size=32, pin_memory=True, num_workers=4, shuffle=True)
val_small_loader = DataLoader(val_small_dataset, batch_size=32, pin_memory=True, num_workers=4)
test_small_loader = DataLoader(test_small_dataset, batch_size=32, pin_memory=True, num_workers=4)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AircraftCNN(num_classes=100)
model = model.to(device)

In [9]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

Epoch 1/20: 100%|██████████| 22/22 [00:12<00:00,  1.80it/s]


✅ Model saved.
Epoch 1: Loss=4.6774, Train Acc=0.0071, Val Acc=0.0157


Epoch 2/20: 100%|██████████| 22/22 [00:11<00:00,  1.86it/s]


✅ Model saved.
Epoch 2: Loss=4.5788, Train Acc=0.0129, Val Acc=0.0171


Epoch 3/20: 100%|██████████| 22/22 [00:10<00:00,  2.15it/s]


...

Epoch 9: Loss=4.2497, Train Acc=0.0657, Val Acc=0.0371


Epoch 10/20: 100%|██████████| 22/22 [00:10<00:00,  2.05it/s]


Epoch 10: Loss=4.2271, Train Acc=0.0729, Val Acc=0.0386


Epoch 11/20: 100%|██████████| 22/22 [00:10<00:00,  2.08it/s]


Early stopping triggered


### Добавим аугментации

In [18]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [12]:
train_small_dataset = AircraftDataset('data/fgvc-aircraft/train_small.csv', 'data/fgvc-aircraft/images', transform)
val_small_dataset = AircraftDataset('data/fgvc-aircraft/val_small.csv', 'data/fgvc-aircraft/images', transform)
test_small_dataset = AircraftDataset('data/fgvc-aircraft/test_small.csv', 'data/fgvc-aircraft/images', transform)

train_small_loader = DataLoader(train_small_dataset, batch_size=32, pin_memory=True, num_workers=4, shuffle=True)
val_small_loader = DataLoader(val_small_dataset, batch_size=32, pin_memory=True, num_workers=4)
test_small_loader = DataLoader(test_small_dataset, batch_size=32, pin_memory=True, num_workers=4)

In [13]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AircraftCNN(num_classes=100)
model = model.to(device)

In [ ]:
train_model(model, train_small_loader, val_small_loader, device, epochs=20)

In [10]:
train_small_dataset = AircraftDataset('data/fgvc-aircraft/train_small.csv', 'data/fgvc-aircraft/images', transform)
val_small_dataset = AircraftDataset('data/fgvc-aircraft/val_small.csv', 'data/fgvc-aircraft/images', transform)
test_small_dataset = AircraftDataset('data/fgvc-aircraft/test_small.csv', 'data/fgvc-aircraft/images', transform)

train_small_loader = DataLoader(train_small_dataset, batch_size=16, pin_memory=True, num_workers=3, shuffle=True)
val_small_loader = DataLoader(val_small_dataset, batch_size=16, pin_memory=True, num_workers=3)
test_small_loader = DataLoader(test_small_dataset, batch_size=16, pin_memory=True, num_workers=3)


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = AircraftCNN(num_classes=100)
model = model.to(device)

In [12]:
train_model(model, train_small_loader, val_small_loader, device, epochs=1)

Epoch 1/1: 100%|██████████| 44/44 [00:11<00:00,  3.85it/s]


✅ Model saved.
Epoch 1: Loss=4.6769, Train Acc=0.0071, Val Acc=0.0171


In [ ]:
device

In [ ]:

model = resnet18(weights='IMAGENET1K_V1')  # предобученные веса
model.fc = nn.Linear(model.fc.in_features, 100)  # подгон под твои классы
model = model.to(device)

In [ ]:

train_model(model, train_small_loader, val_small_loader, device, epochs=20)


### Добавим больше картинок

In [ ]:
train_dataset = AircraftDataset('data/fgvc-aircraft/train.csv', 'data/fgvc-aircraft/images', transform)
val_dataset = AircraftDataset('data/fgvc-aircraft/val.csv', 'data/fgvc-aircraft/images', transform)
test_dataset = AircraftDataset('data/fgvc-aircraft/test.csv', 'data/fgvc-aircraft/images', transform)

train_loader = DataLoader(train_dataset, batch_size=32, pin_memory=True, num_workers=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, pin_memory=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, pin_memory=True, num_workers=4)


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = resnet18(weights='IMAGENET1K_V1')  # предобученные веса
model.fc = nn.Linear(model.fc.in_features, 100)  # подгон под твои классы
model = model.to(device)

# train_model(model, train_loader, val_loader, device, epochs=20)
